# 01. Inputs

All parcels from the verified Preprocessing 01 handoff: annual Sentinel-2 series and Sentinel-1 at the same pixels.

**Input:** `pipeline_outputs/preprocessing/current_quality.json`. **Output:** `pipeline_outputs/improved/current_inputs.json`. **Next:** 02.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
from IPython.display import display
from experiments.improved_v1 import workflow
pd.set_option("display.width", 200)

## Build inputs

In [ ]:
quality_handoff = ""  # Optional quality.json; blank uses the verified Preprocessing 01 pointer
pixels_per_parcel = 5  # 20 matches the main pipeline cap but needs about four times the Sentinel-1 requests
receipt = workflow.build_inputs(root, quality_handoff or None, pixels_per_parcel, progress=lambda m: print(m, flush=True))
print("Inputs receipt:", receipt)

## Parcels and pixels

In [ ]:
from experiments.pilots import data
import numpy as np
rows, arrays = data.load(receipt.parent)
display(pd.DataFrame(rows).groupby(["partition", "crop", "source_dataset"]).partition_group.nunique().rename("parcels").reset_index())
print("Pixels:", len(rows), "| mean clear Sentinel-2 half-months:", round(float(arrays["s2_valid"].sum(1).mean()), 1),
      "| pixels with 20+ Sentinel-1 half-months:", int((np.isfinite(arrays["s1_values"][..., 0]).sum(1) >= 20).sum()))